In [2]:
#!pip install -q transformers accelerate bitsandbytes pandas datasets torch

import torch
print(f"GPU доступен: {torch.cuda.is_available()}")
print(f"Устройство: {torch.cuda.get_device_name(0)}")
print(f"Память: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU доступен: True
Устройство: Tesla T4
Память: 15.6 GB


In [3]:
!pip install -U bitsandbytes>=0.46.1

In [4]:
!pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 88.5 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


## Скачивание датасета

In [5]:
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')

# Это наш выбранный датасет с каггла, в нем данные сразу разбиты на тест и трейн

!mkdir -p ~/.kaggle
!cp /content/drive/MyDrive/kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d xuanhuynh233/ielts-dataset
!unzip ielts-dataset.zip

csv_path = 'train_final.csv'
df = pd.read_csv(csv_path)

print(df.head())
print(f"Размер датасета: {len(df)} строк.")

# Удалим строки, где 'essay' или 'band' отсутствуют,
# или 'band' не являетсня числом.

initial_len = len(df)
df = df.dropna(subset=['essay', 'band'])
df = df[pd.to_numeric(df['band'], errors='coerce').notna()]
df['band'] = df['band'].astype(float) # Убедимся, что band это float
print(f"\nУдалено {initial_len - len(df)} строк с отсутствующими или некорректными 'essay'/'band'.")
print(f"Осталось {len(df)} строк после очистки.")

Mounted at /content/drive
cp: cannot stat '/content/drive/MyDrive/kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/xuanhuynh233/ielts-dataset
License(s): MIT
100% 12.3M/12.3M [00:00<00:00, 51.9MB/s]

Archive:  ielts-dataset.zip
  inflating: test_final.csv          
  inflating: train_final.csv         
                                      band  \
0  7.5\n\n\n\n\n\r\r\r\r\r\r\r\r\r\r\r\r\r   
1  5.0\n\n\n\n\n\r\r\r\r\r\r\r\r\r\r\r\r\r   
2  5.5\n\n\n\n\n\r\r\r\r\r\r\r\r\r\r\r\r\r   
3  5.5\n\n\n\n\n\r\r\r\r\r\r\r\r\r\r\r\r\r   
4    4\n\n\n\n\n\r\r\r\r\r\r\r\r\r\r\r\r\r   

                                              prompt  \
0  Interviews form the basic criteria for most la...   
1  Interviews form the basic selecting criteria f...   
2  Interview form the basic selection criteria fo...   
3  Interviews form the basic selection criteria f...   
4  Interviews from the basi

In [6]:
df.head()

,band,prompt,essay,TR_Band,TR_Comment,CC_Band,CC_Comment,LR_Band,LR_Mistakes,LR_Corrections,LR_Comment,GRA_Band,GRA_Mistakes,GRA_Corrections,GRA_Comment,Overall_Band,General_Feedback,error
0,7.5,Interviews form the basic criteria for most la...,It is believed by some experts that the tradit...,8.0,The essay sufficiently addresses all parts of ...,8.0,The essay is logically structured and the info...,7.0,exams writing; his teamwork skill is measured;...,written exams; their teamwork skills are measu...,The essay demonstrates a sufficient range of v...,7.0,...his ability to do the work and their proble...,...their ability to do the work and their prob...,"A variety of complex structures is used, and t...",7.5,This is a strong response with a very clear st...,NaN
1,5.0,Interviews form the basic selecting criteria f...,Nowadays numerous huge firms allocate an inter...,5.0,The essay addresses the task only partially. I...,5.0,"The essay is presented with some organization,...",5.0,choosing criteria; up-to-the-minute; reap the ...,selection criterion/criteria; effective/reliab...,The writer attempts to use some less common vo...,5.0,having excellent interpersonal skills are; mak...,having excellent interpersonal skills is; effe...,The essay shows a limited range of sentence st...,5.0,The essay presents a clear position and follow...,NaN
2,5.5,Interview form the basic selection criteria fo...,The interview section is the most vital part o...,6.0,"You have addressed all parts of the question, ...",6.0,The essay has a clear structure with an introd...,5.0,Interview section; prospect for the specified ...,The interview stage; prospective candidate for...,You have used an adequate range of vocabulary ...,5.0,Interview form the basic selection criteria; w...,Interviews form the basic selection criteria; ...,You have attempted to use a mix of simple and ...,5.5,Your essay presents a clear opinion and is log...,NaN
3,5.5,Interviews form the basic selection criteria f...,It is argued that the best method to recruit e...,6.0,The essay addresses the prompt by discussing b...,5.5,"The essay is organized into paragraphs, and th...",5.0,acquired by the recruiter; choosing the best e...,possessed by the applicant; choosing the best ...,The range of vocabulary is limited but minimal...,5.0,who do not lacks in any single question; accor...,who do not lack an answer to any single questi...,The writer attempts to use a mix of simple and...,5.5,Your essay has a clear structure and you prese...,NaN
4,4.0,Interviews from the basic selecting criteria f...,Nowadays many companies conduct interviews bef...,4.0,The response addresses the task only in a mini...,4.0,"The essay presents information and ideas, but ...",4.0,chosing; intetviuvs; campaigns; sending them; ...,choosing; interviews; companies; hiring them; ...,The writer uses only basic and repetitive voca...,4.0,Interviews from the basic selecting criteria; ...,Interviews form the basic selection criteria; ...,Only a very limited range of sentence structur...,4.0,"Your essay attempts to address the prompt, but...",NaN


## Загрузка функций

In [7]:
# сразу все импорты и загрузки

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import spearmanr
import re
from tqdm import tqdm
import torch
import time
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

In [8]:
# базовый промт
def create_ielts_prompt(essay_to_grade, example_essays):

    prompt = """You are an expert IELTS Writing Task 2 examiner. Grade essays using IELTS band descriptors (0-9, step 0.5).

IELTS CRITERIA (each 0-9):
1. TR (Task Response): How well the essay addresses the task, presents ideas, and supports arguments
2. CC (Coherence & Cohesion): Organization, paragraph structure, and linking devices
3. LR (Lexical Resource): Vocabulary range, accuracy, and appropriateness
4. GRA (Grammatical Range & Accuracy): Grammar variety, accuracy, and complexity

IMPORTANT:
- Use only 0.5 increments (e.g., 6.0, 6.5, 7.0)
- Overall Band = Average of all 4 criteria
- Provide realistic scores based on IELTS standards

Here are graded examples:

"""

    for i, example in enumerate(example_essays, 1):
        prompt += f"""Example {i}:
Essay: "{example['essay']}..."
TR_Band: {example['TR_Band']}
CC_Band: {example['CC_Band']}
LR_Band: {example['LR_Band']}
GRA_Band: {example['GRA_Band']}
Overall_Band: {example['Overall_Band']}
---

"""

    prompt += f"""Now, grade this essay following IELTS standards:

Essay: "{essay_to_grade}"

Provide your evaluation in this EXACT format:
TR_Band: X.X
CC_Band: X.X
LR_Band: X.X
GRA_Band: X.X
Overall_Band: X.X
Explanation: [Brief justification for each criterion]
"""

    return prompt


In [9]:
# получим все оценки из ответа модели, чтобы считать по ним метрики качества

def extract_ielts_scores(response):

    # Извлечем 5 оценок {'TR': float, 'CC': float, 'LR': float, 'GRA': float, 'Overall': float}

    scores = {}

    patterns = {
        'TR': r'TR[_\s]*Band:\s*(\d+(?:\.\d)?)',
        'CC': r'CC[_\s]*Band:\s*(\d+(?:\.\d)?)',
        'LR': r'LR[_\s]*Band:\s*(\d+(?:\.\d)?)',
        'GRA': r'GRA[_\s]*Band:\s*(\d+(?:\.\d)?)',
        'Overall': r'Overall[_\s]*Band:\s*(\d+(?:\.\d)?)'
    }

    for criterion, pattern in patterns.items():
        match = re.search(pattern, response, re.IGNORECASE)
        if match:
            score = float(match.group(1))
            score = round(score * 2) / 2
            scores[criterion] = min(max(score, 0), 9)
        else:
            scores[criterion] = None

    if scores['Overall'] is None and all(scores[k] is not None for k in ['TR', 'CC', 'LR', 'GRA']):
        avg = np.mean([scores['TR'], scores['CC'], scores['LR'], scores['GRA']])
        scores['Overall'] = round(avg * 2) / 2

    return scores

In [10]:
# функция оценки эссе по нашему промту
def grade_ielts_essay(essay, example_essays, model, tokenizer, max_tokens=600):


    prompt = create_ielts_prompt(essay, example_essays)
    # задаем роль
    messages = [
        {"role": "system", "content": "You are an expert IELTS Writing examiner with years of experience."},
        {"role": "user", "content": prompt}
    ]


    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.2,  # Очень низкая для стабильности, была 0.3, выдавала разницу в качестве в 1 единцу MAE
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1
        )

    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )

    return response


In [11]:
# оцениваем качество модели

def calculate_ielts_metrics(actual, predicted):

    mask = ~(pd.isna(actual) | pd.isna(predicted))
    actual = np.array(actual)[mask]
    predicted = np.array(predicted)[mask]

    if len(actual) == 0:
        return None

    # MAE
    mae = mean_absolute_error(actual, predicted)

    # RMSE
    rmse = np.sqrt(mean_squared_error(actual, predicted))

    # MBE чтобы понимать занижение/завышение оценок
    mbe = np.mean(predicted - actual)

    # Корреляция Спирмена
    correlation, _ = spearmanr(actual, predicted)

    # Точность
    acc_05 = np.mean(np.abs(predicted - actual) <= 0.5)
    acc_10 = np.mean(np.abs(predicted - actual) <= 1.0)

    # Процент точных совпадений (особенно важно для 0.5 шага)
    exact_match = np.mean(predicted == actual)

    # QWK
    from sklearn.metrics import cohen_kappa_score
    qwk = cohen_kappa_score(
        (actual * 2).astype(int),  # Переводим в целые (умножаем на 2)
        (predicted * 2).astype(int),
        weights='quadratic'
    )

    return {
        'MAE': mae,
        'RMSE': rmse,
        'MBE': mbe,
        'Correlation': correlation,
        'QWK': qwk,
        'Accuracy_±0.5': acc_05,
        'Accuracy_±1.0': acc_10,
        'Exact_Match': exact_match,
        'n_samples': len(actual)
    }

## Сравнение разных моделей

In [ ]:
#!pip uninstall -y bitsandbytes
#!pip install bitsandbytes>=0.46.1

Found existing installation: bitsandbytes 0.49.2
Uninstalling bitsandbytes-0.49.2:
  Successfully uninstalled bitsandbytes-0.49.2


In [12]:
# перечисляю модели для сравнения
model_names = [
    #"microsoft/Phi-3-mini-128k-instruct",
    #"unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    #"Qwen/Qwen2.5-3B-Instruct",
    "Qwen/Qwen2.5-7B-Instruct"
]

name_map = {
    "microsoft/Phi-3-mini-128k-instruct":'phi',
    "unsloth/Llama-3.2-3B-Instruct-bnb-4bit":'llama-3b',
    "Qwen/Qwen2.5-7B-Instruct":'qwen_7b',
    "Qwen/Qwen2.5-3B-Instruct": "qwen-3b"

}


sample_size = 30
test_df = df.sample(sample_size, random_state=42).copy()

models_time = {
    'Model_name': [],
    'Model_load_time (sec)':[],
    'Model_load_memory (MB)':[],
    'Avg_inference_time (sec)': [],
    'Total_inference_time (sec)': []
}

# будем давать 5 примеров оценненых эссе внутри промта
examples = df[~df.index.isin(test_df.index)].sample(5).to_dict('records')

print(f"Оценка {sample_size} IELTS эссе...")
print()

for current_model_name in model_names:
    model_alias = name_map[current_model_name]

    predictions = {
        'TR': [],
        'CC': [],
        'LR': [],
        'GRA': [],
        'Overall': []
    }
    explanations = []

    print(f'Начало загрузки модели {current_model_name}')

    # Конфигурация 4-bit квантизации
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )

    current_model = AutoModelForCausalLM.from_pretrained(
    current_model_name,
    quantization_config=bnb_config,
    device_map="auto"
    )

    tokenizer = AutoTokenizer.from_pretrained(current_model_name)
    start_time = time.time()
    start_memory = torch.cuda.memory_allocated() if torch.cuda.is_available() else psutil.virtual_memory().used

    current_model = AutoModelForCausalLM.from_pretrained(
      current_model_name,
      quantization_config=bnb_config,
      device_map="auto"
    )

    load_time = time.time() - start_time
    end_memory = torch.cuda.memory_allocated() if torch.cuda.is_available() else psutil.virtual_memory().used
    memory_used_mb = (end_memory - start_memory) / (1024 ** 2)

    models_time['Model_name'].append(current_model_name)
    models_time['Model_load_time (sec)'].append(load_time)
    models_time['Model_load_memory (MB)'].append(memory_used_mb)
    print(f'Окончание загрузки модели {current_model}')

    inference_times = []

    for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
        start_infer = time.time()
        try:
            response = grade_ielts_essay(
                row['essay'],
                examples,
                current_model,
                tokenizer,
                max_tokens=600
            )
            scores = extract_ielts_scores(response)

            for criterion in ['TR', 'CC', 'LR', 'GRA', 'Overall']:
                predictions[criterion].append(scores.get(criterion, None))

            explanations.append(response)

        except Exception as e:
            print(f"Ошибка на эссе {idx}: {e}")
            for criterion in predictions.keys():
                predictions[criterion].append(None)
            explanations.append("")

        inference_times.append(time.time() - start_infer)

    # Статистика по времени
    avg_inference_time = np.mean(inference_times) if inference_times else 0.0
    sum_inference_time = np.sum(inference_times)
    models_time['Avg_inference_time (sec)'].append(avg_inference_time)
    models_time['Total_inference_time (sec)'].append(sum_inference_time)

    for criterion in ['TR', 'CC', 'LR', 'GRA', 'Overall']:
        test_df[f'pred_{criterion}_{model_alias}'] = predictions[criterion]

    test_df[f'explanation_{model_alias}'] = explanations
    print(f"Модель {model_alias} обработана")

    del current_model, tokenizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("Оценка завершена")

Оценка 30 IELTS эссе...

Начало загрузки модели Qwen/Qwen2.5-7B-Instruct


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Окончание загрузки модели Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear4bit(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear4bit(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear4bit(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear4bit(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear4bit(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    


100%|██████████| 30/30 [29:01<00:00, 58.05s/it]

Модель qwen_7b обработана
Оценка завершена


In [13]:
# Датасет с временем работы моделей
models_time_df = pd.DataFrame(models_time)
models_time_df = models_time_df.round(3)
models_time_df = models_time_df[['Model_name', 'Model_load_time (sec)', 'Avg_inference_time (sec)', 'Total_inference_time (sec)']]
models_time_df

,Model_name,Model_load_time (sec),Avg_inference_time (sec),Total_inference_time (sec)
0,Qwen/Qwen2.5-7B-Instruct,55.595,58.044,1741.324


In [14]:
models_time_df_old = pd.read_csv('models_time_df_model_comparison_3.csv')
models_time_df_new = pd.concat([models_time_df_old, models_time_df])
models_time_df_new.to_csv('models_time_df_new.csv', index=False)
models_time_df_new.head()

,Model_name,Model_load_time (sec),Avg_inference_time (sec),Total_inference_time (sec)
0,microsoft/Phi-3-mini-128k-instruct,24.499,40.479,1214.368
1,unsloth/Llama-3.2-3B-Instruct-bnb-4bit,4.443,38.620,1158.601
2,Qwen/Qwen2.5-3B-Instruct,22.974,47.999,1439.979
0,Qwen/Qwen2.5-7B-Instruct,55.595,58.044,1741.324


In [15]:
print("Считаю метрики качества")

criteria_mapping = {
    'TR': 'TR_Band',
    'CC': 'CC_Band',
    'LR': 'LR_Band',
    'GRA': 'GRA_Band',
    'Overall': 'Overall_Band'
}

all_metrics = {'Criterion':[], 'Model_name':[],'MAE':[], 'RMSE':[], 'MBE':[]
               , 'Correlation':[], 'QWK':[], 'Accuracy_±0.5':[], 'Accuracy_±1.0':[], 'Exact_Match':[], 'n_samples':[]}

for current_model_name in model_names[:3]:
  print(current_model_name)
  model_alias = name_map[current_model_name]
  for criterion, actual_col in criteria_mapping.items():
      actual = test_df[actual_col]
      predicted = test_df[f'pred_{criterion}_{model_alias}']

      metrics = calculate_ielts_metrics(actual, predicted)

      if metrics:
        all_metrics['Criterion'].append(criterion)
        all_metrics['Model_name'].append(current_model_name)
        all_metrics['MAE'].append(round(metrics['MAE'], 3))
        all_metrics['RMSE'].append(round(metrics['RMSE'], 3))
        all_metrics['MBE'].append(round(metrics['MBE'], 3))
        all_metrics['Correlation'].append(round(metrics['Correlation'], 3))
        all_metrics['QWK'].append(round(metrics['QWK'], 3))
        all_metrics['Accuracy_±0.5'].append(round(metrics['Accuracy_±0.5'], 3))
        all_metrics['Accuracy_±1.0'].append(round(metrics['Accuracy_±1.0'], 3))
        all_metrics['Exact_Match'].append(round(metrics['Exact_Match'], 3))
        all_metrics['n_samples'].append(round(metrics['n_samples'], 3))

df_all_metrics = pd.DataFrame(all_metrics)
df_all_metrics

Считаю метрики качества
Qwen/Qwen2.5-7B-Instruct


,Criterion,Model_name,MAE,RMSE,MBE,Correlation,QWK,Accuracy_±0.5,Accuracy_±1.0,Exact_Match,n_samples
0,TR,Qwen/Qwen2.5-7B-Instruct,1.533,1.871,1.433,-0.080,0.020,0.400,0.467,0.033,30
1,CC,Qwen/Qwen2.5-7B-Instruct,1.317,1.599,1.050,0.302,0.132,0.367,0.467,0.100,30
2,LR,Qwen/Qwen2.5-7B-Instruct,1.733,2.045,1.667,0.219,0.060,0.333,0.333,0.067,30
3,GRA,Qwen/Qwen2.5-7B-Instruct,1.683,1.998,1.617,0.245,0.056,0.333,0.367,0.067,30
4,Overall,Qwen/Qwen2.5-7B-Instruct,1.450,1.846,1.417,0.156,0.053,0.367,0.467,0.233,30


In [16]:
df_all_metrics
df_all_metrics_old = pd.read_csv('df_all_metrics_model_comparison_3.csv')
df_all_metrics_new = pd.concat([df_all_metrics_old, df_all_metrics])
df_all_metrics_new.to_csv('df_all_metrics_new.csv', index=False)
df_all_metrics_new.head()

,Criterion,Model_name,MAE,RMSE,MBE,Correlation,QWK,Accuracy_±0.5,Accuracy_±1.0,Exact_Match,n_samples
0,TR,microsoft/Phi-3-mini-128k-instruct,1.333,1.673,1.267,0.405,0.139,0.333,0.467,0.167,30
1,CC,microsoft/Phi-3-mini-128k-instruct,1.533,1.839,1.367,0.358,0.161,0.233,0.433,0.067,30
2,LR,microsoft/Phi-3-mini-128k-instruct,1.633,1.906,1.567,0.450,0.163,0.233,0.367,0.100,30
3,GRA,microsoft/Phi-3-mini-128k-instruct,1.717,1.947,1.583,0.393,0.105,0.133,0.367,0.033,30
4,Overall,microsoft/Phi-3-mini-128k-instruct,1.450,1.749,1.350,0.418,0.131,0.300,0.433,0.100,30


In [17]:
# финальный датасет
df_comparison = pd.merge(df_all_metrics_new, models_time_df_new, left_on = 'Model_name', right_on = 'Model_name', how = 'inner')
df_comparison

,Criterion,Model_name,MAE,RMSE,MBE,Correlation,QWK,Accuracy_±0.5,Accuracy_±1.0,Exact_Match,n_samples,Model_load_time (sec),Avg_inference_time (sec),Total_inference_time (sec)
0,TR,microsoft/Phi-3-mini-128k-instruct,1.333,1.673,1.267,0.405,0.139,0.333,0.467,0.167,30,24.499,40.479,1214.368
1,CC,microsoft/Phi-3-mini-128k-instruct,1.533,1.839,1.367,0.358,0.161,0.233,0.433,0.067,30,24.499,40.479,1214.368
2,LR,microsoft/Phi-3-mini-128k-instruct,1.633,1.906,1.567,0.450,0.163,0.233,0.367,0.100,30,24.499,40.479,1214.368
3,GRA,microsoft/Phi-3-mini-128k-instruct,1.717,1.947,1.583,0.393,0.105,0.133,0.367,0.033,30,24.499,40.479,1214.368
4,Overall,microsoft/Phi-3-mini-128k-instruct,1.450,1.749,1.350,0.418,0.131,0.300,0.433,0.100,30,24.499,40.479,1214.368
5,TR,unsloth/Llama-3.2-3B-Instruct-bnb-4bit,1.400,1.703,1.300,0.352,0.152,0.267,0.500,0.133,30,4.443,38.620,1158.601
6,CC,unsloth/Llama-3.2-3B-Instruct-bnb-4bit,1.533,1.844,1.367,0.365,0.203,0.267,0.467,0.033,30,4.443,38.620,1158.601
7,LR,unsloth/Llama-3.2-3B-Instruct-bnb-4bit,1.417,1.739,1.250,0.301,0.203,0.267,0.400,0.200,30,4.443,38.620,1158.601
8,GRA,unsloth/Llama-3.2-3B-Instruct-bnb-4bit,1.750,2.035,1.650,0.291,0.099,0.233,0.367,0.033,30,4.443,38.620,1158.601
9,Overall,unsloth/Llama-3.2-3B-Instruct-bnb-4bit,1.433,1.708,1.267,0.341,0.154,0.333,0.433,0.033,30,4.443,38.620,1158.601


## Финальные датасеты со сравнениями

In [18]:
df_all_metrics.loc[df_all_metrics['Criterion'] == 'Overall']

,Criterion,Model_name,MAE,RMSE,MBE,Correlation,QWK,Accuracy_±0.5,Accuracy_±1.0,Exact_Match,n_samples
4,Overall,Qwen/Qwen2.5-7B-Instruct,1.45,1.846,1.417,0.156,0.053,0.367,0.467,0.233,30


In [19]:
# финальное сравнение моделей по Overall
df_comparison.loc[df_comparison['Criterion'] == 'Overall']

,Criterion,Model_name,MAE,RMSE,MBE,Correlation,QWK,Accuracy_±0.5,Accuracy_±1.0,Exact_Match,n_samples,Model_load_time (sec),Avg_inference_time (sec),Total_inference_time (sec)
4,Overall,microsoft/Phi-3-mini-128k-instruct,1.450,1.749,1.350,0.418,0.131,0.300,0.433,0.100,30,24.499,40.479,1214.368
9,Overall,unsloth/Llama-3.2-3B-Instruct-bnb-4bit,1.433,1.708,1.267,0.341,0.154,0.333,0.433,0.033,30,4.443,38.620,1158.601
14,Overall,Qwen/Qwen2.5-3B-Instruct,1.367,1.638,1.133,0.005,0.021,0.367,0.467,0.067,30,22.974,47.999,1439.979
19,Overall,Qwen/Qwen2.5-7B-Instruct,1.450,1.846,1.417,0.156,0.053,0.367,0.467,0.233,30,55.595,58.044,1741.324


In [20]:
# финальное сравнение моделей по TR
df_comparison.loc[df_comparison['Criterion'] == 'TR']

,Criterion,Model_name,MAE,RMSE,MBE,Correlation,QWK,Accuracy_±0.5,Accuracy_±1.0,Exact_Match,n_samples,Model_load_time (sec),Avg_inference_time (sec),Total_inference_time (sec)
0,TR,microsoft/Phi-3-mini-128k-instruct,1.333,1.673,1.267,0.405,0.139,0.333,0.467,0.167,30,24.499,40.479,1214.368
5,TR,unsloth/Llama-3.2-3B-Instruct-bnb-4bit,1.400,1.703,1.300,0.352,0.152,0.267,0.500,0.133,30,4.443,38.620,1158.601
10,TR,Qwen/Qwen2.5-3B-Instruct,1.217,1.519,1.050,0.323,0.073,0.333,0.533,0.233,30,22.974,47.999,1439.979
15,TR,Qwen/Qwen2.5-7B-Instruct,1.533,1.871,1.433,-0.080,0.020,0.400,0.467,0.033,30,55.595,58.044,1741.324


In [21]:
# финальное сравнение моделей по CC
df_comparison.loc[df_comparison['Criterion'] == 'CC']

,Criterion,Model_name,MAE,RMSE,MBE,Correlation,QWK,Accuracy_±0.5,Accuracy_±1.0,Exact_Match,n_samples,Model_load_time (sec),Avg_inference_time (sec),Total_inference_time (sec)
1,CC,microsoft/Phi-3-mini-128k-instruct,1.533,1.839,1.367,0.358,0.161,0.233,0.433,0.067,30,24.499,40.479,1214.368
6,CC,unsloth/Llama-3.2-3B-Instruct-bnb-4bit,1.533,1.844,1.367,0.365,0.203,0.267,0.467,0.033,30,4.443,38.620,1158.601
11,CC,Qwen/Qwen2.5-3B-Instruct,1.483,1.810,1.250,-0.030,0.001,0.300,0.433,0.133,30,22.974,47.999,1439.979
16,CC,Qwen/Qwen2.5-7B-Instruct,1.317,1.599,1.050,0.302,0.132,0.367,0.467,0.100,30,55.595,58.044,1741.324


In [22]:
# финальное сравнение моделей по LR
df_comparison.loc[df_comparison['Criterion'] == 'LR']

,Criterion,Model_name,MAE,RMSE,MBE,Correlation,QWK,Accuracy_±0.5,Accuracy_±1.0,Exact_Match,n_samples,Model_load_time (sec),Avg_inference_time (sec),Total_inference_time (sec)
2,LR,microsoft/Phi-3-mini-128k-instruct,1.633,1.906,1.567,0.450,0.163,0.233,0.367,0.100,30,24.499,40.479,1214.368
7,LR,unsloth/Llama-3.2-3B-Instruct-bnb-4bit,1.417,1.739,1.250,0.301,0.203,0.267,0.400,0.200,30,4.443,38.620,1158.601
12,LR,Qwen/Qwen2.5-3B-Instruct,1.517,1.782,1.317,0.208,0.028,0.233,0.467,0.100,30,22.974,47.999,1439.979
17,LR,Qwen/Qwen2.5-7B-Instruct,1.733,2.045,1.667,0.219,0.060,0.333,0.333,0.067,30,55.595,58.044,1741.324


In [23]:
# финальное сравнение моделей по GRA
df_comparison.loc[df_comparison['Criterion'] == 'GRA']

,Criterion,Model_name,MAE,RMSE,MBE,Correlation,QWK,Accuracy_±0.5,Accuracy_±1.0,Exact_Match,n_samples,Model_load_time (sec),Avg_inference_time (sec),Total_inference_time (sec)
3,GRA,microsoft/Phi-3-mini-128k-instruct,1.717,1.947,1.583,0.393,0.105,0.133,0.367,0.033,30,24.499,40.479,1214.368
8,GRA,unsloth/Llama-3.2-3B-Instruct-bnb-4bit,1.750,2.035,1.650,0.291,0.099,0.233,0.367,0.033,30,4.443,38.620,1158.601
13,GRA,Qwen/Qwen2.5-3B-Instruct,1.500,1.770,1.333,0.222,0.039,0.200,0.467,0.133,30,22.974,47.999,1439.979
18,GRA,Qwen/Qwen2.5-7B-Instruct,1.683,1.998,1.617,0.245,0.056,0.333,0.367,0.067,30,55.595,58.044,1741.324
